# Retail Key Account Profitability Analysis
### SuperStore US — H1 2015 (January – June 2015)

---

**Objective:** Evaluate customer profitability, uncover discount leakage, classify accounts by strategic value, and identify margin pressure by product category and region.

**Dataset:** SuperStore US | 1,952 transactions | 1,130 unique customers | 4 regions | 3 product categories

**Scope:**
- Data cleaning & validation
- KPI engineering (profit margin, discount rate)
- Customer-level profitability aggregation
- Strategic account tier classification
- Discount leakage detection
- Profit concentration (Pareto) analysis
- Category-level margin analysis
- Executive summary & recommendations

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('SuperStoreUS_data.xlsx', sheet_name='Orders')

print('Dataset shape    :', df.shape)
print('Date range       :', df['Order Date'].min(), 'to', df['Order Date'].max())
print('Unique Customers :', df['Customer Name'].nunique())
print('Unique Orders    :', df['Order ID'].nunique())
df.head(3)

## 2. Data Inspection & Quality Check

Checking data types, missing values, and duplicates before any analysis.

In [ ]:
df.info()
print('\nMissing Values:')
print(df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())

## 3. Data Cleaning

- Parsed date columns to datetime for any future time-based operations
- Cast Postal Code to string to preserve leading zeros
- Dropped non-essential columns only — **Customer Name and Order ID retained** for groupby and aggregation
- No missing values found — no imputation needed

In [ ]:
if not pd.api.types.is_datetime64_any_dtype(df['Order Date']):
    df['Order Date'] = pd.to_datetime(df['Order Date'], unit='d', origin='1899-12-30')
if not pd.api.types.is_datetime64_any_dtype(df['Ship Date']):
    df['Ship Date'] = pd.to_datetime(df['Ship Date'], unit='d', origin='1899-12-30')

df['Postal Code'] = df['Postal Code'].astype(str)

# Drop only non-essential columns — Customer Name and Order ID must be kept
df = df.drop(columns=['Row ID', 'Product Container', 'Ship Mode'], errors='ignore')

print('After cleaning shape :', df.shape)
print('Customer Name present:', 'Customer Name' in df.columns)  # must be True
print('Order ID present     :', 'Order ID' in df.columns)       # must be True

## 4. KPI Engineering

Creating transaction-level metrics and establishing portfolio-level benchmarks.

**Key distinction:** Portfolio Margin = Total Profit / Total Sales = **11.64%**
This is used as the margin benchmark throughout — NOT the transaction-level average
(which is distorted by extreme loss transactions).

In [ ]:
df['Profit_Margin_Pct'] = (df['Profit'] / df['Sales']) * 100
df['Discount_Pct']      = df['Discount'] * 100
df['Profit_Per_Unit']   = df['Profit'] / df['Quantity ordered new']
df['Sales_Per_Unit']    = df['Sales']  / df['Quantity ordered new']

# Portfolio-level benchmarks — used as thresholds throughout analysis
# Portfolio Margin = Total Profit / Total Sales (not transaction average)
PORTFOLIO_MARGIN = round((df['Profit'].sum() / df['Sales'].sum()) * 100, 2)  # 11.64%
AVG_DISCOUNT     = round(df['Discount'].mean(), 4)                            # 0.049

print(f'Total Sales             : ${df["Sales"].sum():,.0f}')
print(f'Total Profit            : ${df["Profit"].sum():,.0f}')
print(f'Portfolio Profit Margin : {PORTFOLIO_MARGIN}%   <- benchmark for leakage detection')
print(f'Average Discount Rate   : {round(AVG_DISCOUNT*100,2)}%  <- benchmark for discount flagging')
print(f'Loss Transactions       : {(df["Profit"]<0).sum()} ({round((df["Profit"]<0).sum()*100/len(df),2)}%)')

## 5. Customer-Level Profitability Aggregation

Aggregating by **Customer Name** to build one profitability row per customer.

**Note:** Grouping by Customer Name (not Customer ID) gives **1,130 unique customers**,
consistent with the dashboard. Grouping by Customer ID may yield slightly higher counts
due to multi-region transactions.

**Overall_Margin_Pct** = Total_Profit / Total_Sales per customer — this is the correct
customer-level margin, not an average of transaction margins.

In [ ]:
customer_df = df.groupby(['Customer Name', 'Customer Segment', 'Region']).agg(
    Total_Sales       = ('Sales',                'sum'),
    Total_Profit      = ('Profit',               'sum'),
    Total_Orders      = ('Order ID',             'nunique'),
    Total_Units       = ('Quantity ordered new', 'sum'),
    Avg_Discount_Rate = ('Discount',             'mean'),
).reset_index()

# Portfolio-level margin per customer (Total Profit / Total Sales)
customer_df['Overall_Margin_Pct'] = (
    customer_df['Total_Profit'] / customer_df['Total_Sales']
) * 100

customer_df = customer_df.round(2)

print('Total Unique Customers :', customer_df.shape[0])          # 1,130
print('Loss-Making Customers  :', (customer_df['Overall_Margin_Pct'] < 0).sum())  # 539
print('Profitable Customers   :', (customer_df['Overall_Margin_Pct'] > 0).sum())  # 591
customer_df.describe().round(2)

## 6. Strategic Account Tier Classification

Customers classified into 4 tiers based on revenue quartile and profit margin:

| Tier | Criteria | Count | Action |
|------|----------|-------|--------|
| **Strategic Core** | Q4 revenue + margin > 15% | 151 | Protect & grow |
| **Margin Risk** | Q4 revenue + margin ≤ 15% | 58 | Pricing audit |
| **Loss-Making** | Negative overall margin | 567 | Immediate review |
| **Standard Account** | All others | 415 | Routine management |

**Insight:** 567 out of 1,191 customer-region combinations are loss-making. A small portion
fall into Strategic Core — these accounts must be protected with formal KAM coverage.

In [ ]:
customer_df['Revenue_Quartile'] = pd.qcut(
    customer_df['Total_Sales'], 4,
    labels=['Q1_Low', 'Q2_MidLow', 'Q3_MidHigh', 'Q4_High']
)

def classify_account(row):
    if row['Overall_Margin_Pct'] < 0:
        return 'Loss-Making'
    elif row['Revenue_Quartile'] == 'Q4_High' and row['Overall_Margin_Pct'] > 15:
        return 'Strategic Core'
    elif row['Revenue_Quartile'] == 'Q4_High' and row['Overall_Margin_Pct'] <= 15:
        return 'Margin Risk'
    else:
        return 'Standard Account'

customer_df['Strategic_Tier'] = customer_df.apply(classify_account, axis=1)

print('Strategic Tier Distribution:')
print(customer_df['Strategic_Tier'].value_counts())
# Expected: Loss-Making=567, Standard=415, Strategic Core=151, Margin Risk=58

In [ ]:
tier_summary = customer_df.groupby('Strategic_Tier').agg(
    Customer_Count    = ('Customer Name',     'count'),
    Total_Revenue     = ('Total_Sales',       'sum'),
    Avg_Margin_Pct    = ('Overall_Margin_Pct','mean'),
    Avg_Discount_Rate = ('Avg_Discount_Rate', 'mean')
).reset_index().round(2)

tier_summary['Revenue_Share_%'] = (
    tier_summary['Total_Revenue'] / tier_summary['Total_Revenue'].sum() * 100
).round(1)

print(tier_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=tier_summary, x='Strategic_Tier', y='Customer_Count',
            palette='Blues_d', ax=axes[0])
axes[0].set_title('Customer Count by Strategic Tier')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20)

sns.barplot(data=tier_summary, x='Strategic_Tier', y='Avg_Margin_Pct',
            palette='RdYlGn', ax=axes[1])
axes[1].set_title('Avg Profit Margin % by Strategic Tier')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20)

plt.tight_layout()
plt.show()

## 7. Discount Leakage Detection

Identifying customers with **above-average discount** AND **below-portfolio margin** —
the primary signal of discount governance failure.

**Thresholds used:**
- Discount threshold : AVG_DISCOUNT = 4.9%
- Margin threshold   : PORTFOLIO_MARGIN = 11.64%

**Expected findings:**
- ~397 flagged accounts
- These accounts represent ~30.5% of total portfolio revenue
- Technology shows deepest losses in high-risk accounts (-36.21% margin)

**Note:** Revenue exposure at ~30% signals an urgent need for discount governance policy.

In [ ]:
risk_customers = customer_df[
    (customer_df['Avg_Discount_Rate'] > AVG_DISCOUNT) &
    (customer_df['Overall_Margin_Pct'] < PORTFOLIO_MARGIN)
].sort_values('Avg_Discount_Rate', ascending=False)

total_revenue    = customer_df['Total_Sales'].sum()
risk_revenue     = risk_customers['Total_Sales'].sum()
risk_revenue_pct = round(risk_revenue / total_revenue * 100, 2)

print(f'High-Risk Customers    : {len(risk_customers)}')       # 397
print(f'Total Portfolio Revenue: ${total_revenue:,.0f}')       # $1,924,338
print(f'Revenue at Risk        : ${risk_revenue:,.0f} ({risk_revenue_pct}%)')  # ~30.5%
print()
print('Top 10 High-Discount Low-Margin Customers:')
risk_customers[['Customer Name','Customer Segment','Region',
                'Total_Sales','Total_Profit','Avg_Discount_Rate',
                'Overall_Margin_Pct']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=customer_df, x='Strategic_Tier', y='Avg_Discount_Rate',
            palette='Set2', ax=axes[0])
axes[0].axhline(AVG_DISCOUNT, color='red', linestyle='--', label=f'Portfolio Avg {round(AVG_DISCOUNT*100,1)}%')
axes[0].set_title('Discount Distribution by Strategic Tier')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20)
axes[0].legend()

values = [total_revenue - risk_revenue, risk_revenue]
labels = [f'Standard\n${total_revenue-risk_revenue:,.0f}',
          f'High-Risk\n${risk_revenue:,.0f} ({risk_revenue_pct}%)']
axes[1].pie(values, labels=labels, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c'], startangle=90)
axes[1].set_title('Revenue Split: Standard vs High-Risk Customers')

plt.tight_layout()
plt.show()

## 8. Profit Concentration — Pareto Analysis

Determining what percentage of customers generate 80% of total profit.

**Expected finding:** Top **26%** of profitable customers (162 out of 624) generate **80% of profit**.

**Business implication:** Losing even 5-10 of these accounts creates a significant P&L impact.
Structured KAM retention strategy is critical for this group.

In [ ]:
profit_positive = customer_df[customer_df['Total_Profit'] > 0].copy()
profit_positive = profit_positive.sort_values('Total_Profit', ascending=False).reset_index(drop=True)

total_positive_profit = profit_positive['Total_Profit'].sum()
profit_positive['Cumulative_Profit'] = profit_positive['Total_Profit'].cumsum()
profit_positive['Cumulative_Pct']    = (
    profit_positive['Cumulative_Profit'] / total_positive_profit * 100
)

idx_80          = (profit_positive['Cumulative_Pct'] >= 80).idxmax()
customers_80pct = idx_80 + 1
pct_of_base     = round(customers_80pct / len(profit_positive) * 100, 1)

print(f'Total profitable customers  : {len(profit_positive)}')   # 624
print(f'Customers for 80%% profit   : {customers_80pct} ({pct_of_base}%)')  # 162 (26%)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(profit_positive)+1),
         profit_positive['Cumulative_Pct'], color='steelblue', linewidth=2)
plt.axhline(80, color='red', linestyle='--', label='80% threshold')
plt.axvline(customers_80pct, color='orange', linestyle='--',
            label=f'{customers_80pct} customers = {pct_of_base}% of profitable base')
plt.fill_between(range(1, customers_80pct+1),
                 profit_positive['Cumulative_Pct'].iloc[:customers_80pct],
                 alpha=0.15, color='orange')
plt.title('Cumulative Profit Concentration — Pareto Analysis')
plt.xlabel('Customer Rank (by Profit)')
plt.ylabel('Cumulative Profit %')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Category-Level Margin Analysis

Comparing structural margin across 3 product categories:

| Category | Margin % | Revenue Share | Insight |
|----------|----------|---------------|---------|
| Office Supplies | 16.24% | 28.7% | Best margin — protect from discounting |
| Technology | 10.57% | 37.0% | Below portfolio avg — discount governance needed |
| Furniture | 8.97% | 34.3% | Structurally lowest — pricing floor required |

**Key insight:** Furniture margin pressure is structural (low base margin).
Technology margin pressure is governance-driven (excessive discounting).
Response strategy is fundamentally different for each.

In [ ]:
category_summary = df.groupby('Product Category').agg(
    Total_Sales  = ('Sales',    'sum'),
    Total_Profit = ('Profit',   'sum'),
    Avg_Discount = ('Discount', 'mean'),
    Transactions = ('Sales',    'count')
).reset_index()

category_summary['Profit_Margin_%'] = (
    category_summary['Total_Profit'] / category_summary['Total_Sales'] * 100
).round(2)
category_summary['Revenue_Share_%'] = (
    category_summary['Total_Sales'] / category_summary['Total_Sales'].sum() * 100
).round(1)

print(category_summary[['Product Category','Total_Sales','Total_Profit',
                         'Profit_Margin_%','Revenue_Share_%']].to_string(index=False))
# Office Supplies: 16.24% | Technology: 10.57% | Furniture: 8.97%

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#e74c3c' if m < 10 else '#f39c12' if m < 15 else '#2ecc71'
          for m in category_summary['Profit_Margin_%']]
axes[0].bar(category_summary['Product Category'], category_summary['Profit_Margin_%'], color=colors)
axes[0].axhline(PORTFOLIO_MARGIN, color='blue', linestyle='--',
                label=f'Portfolio avg {PORTFOLIO_MARGIN}%')
axes[0].set_title('Profit Margin % by Product Category')
axes[0].set_ylabel('Profit Margin %')
axes[0].legend()

axes[1].bar(category_summary['Product Category'], category_summary['Revenue_Share_%'],
            color='steelblue')
axes[1].set_title('Revenue Share % by Product Category')
axes[1].set_ylabel('Revenue Share %')

plt.tight_layout()
plt.show()

## 10. Category Exposure in High-Risk Accounts

Examining which categories generate the deepest losses within the 397 high-risk customers.

**Expected finding:** All 3 categories show negative margins in high-risk accounts:
- Technology: **-36.21%** (deepest loss)
- Furniture: **-19.16%**
- Office Supplies: **-15.63%**

**Conclusion:** Leakage is NOT structural — it is governance-driven.
All categories go negative only in high-discount accounts. Fix = discount governance, not pricing restructure.

In [ ]:
risk_ids  = risk_customers['Customer Name'].unique()
risk_data = df[df['Customer Name'].isin(risk_ids)]

risk_cat = risk_data.groupby('Product Category').agg(
    Total_Sales  = ('Sales',  'sum'),
    Total_Profit = ('Profit', 'sum'),
    Transactions = ('Sales',  'count')
).reset_index()

risk_cat['Profit_Margin_%'] = (
    risk_cat['Total_Profit'] / risk_cat['Total_Sales'] * 100
).round(2)

print('Category Margins — High-Risk Customers Only:')
print(risk_cat[['Product Category','Total_Sales','Total_Profit','Profit_Margin_%']].to_string(index=False))
# Technology: -36.21% | Furniture: -19.16% | Office Supplies: -15.63%

plt.figure(figsize=(7, 4))
colors = ['#e74c3c' if m < 0 else '#f39c12' for m in risk_cat['Profit_Margin_%']]
plt.bar(risk_cat['Product Category'], risk_cat['Profit_Margin_%'], color=colors)
plt.axhline(0, color='black', linewidth=1)
plt.title('Profit Margin % by Category — High-Risk Customers (All Negative)')
plt.ylabel('Profit Margin %')
plt.tight_layout()
plt.show()

## 11. Executive Summary

---

### Key Findings

| # | Finding | Metric | Implication |
|---|---------|--------|-------------|
| 1 | Profit highly concentrated | Top 26% customers = 80% profit (162 of 624) | High retention risk |
| 2 | Discount leakage present | 397 high-risk accounts = 30.5% of revenue | Urgent governance needed |
| 3 | Furniture structurally thin | 8.97% margin — lowest of 3 categories | Pricing floor required |
| 4 | Technology deepest loss in risk accounts | -36.21% margin in high-risk accounts | Discount cap needed |
| 5 | Office Supplies strongest margin | 16.24% — above portfolio avg | Protect from commoditization |
| 6 | South region net loss | -4.04% margin, 52.71% loss transactions | Immediate review |

### Strategic Recommendations

1. **Tier-Based Discount Governance** — Loss-Making and Margin Risk accounts must not receive further discounts without manager approval
2. **Protect Strategic Core (151 accounts)** — Assign formal KAM coverage; quarterly retention reviews
3. **Technology Discount Cap** — All Technology discounts > 5% in high-risk accounts require manager sign-off
4. **Furniture Margin Floor** — Minimum 10% margin floor; discount > 8% requires written justification
5. **Incentive Realignment** — Align sales incentives to margin contribution, not top-line revenue
6. **South Region Audit** — Immediate discount policy review; Loss_Transaction_Pct at 52.71% is highest of all regions

### Business Impact

> Reducing discount leakage in the 397 high-risk accounts (30.5% of revenue) can significantly improve portfolio margin from 11.64% toward the 15%+ industry benchmark — without sacrificing revenue growth.